### #9

Kaggle competition: [\[link\]](https://www.kaggle.com/competitions/playground-series-s5e9/)

Entry by Robin R.P.M. Kras

robkras.com

### ⭐ 1. Introduction & Overview


Your Goal: The goal of this competition is to predict a song's beats-per-minute.

### 🔹 2. Import Libraries & Set Up


In [37]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-

# =====================
# General utilities
# =====================
import json
import os
import pickle
import time
from collections import Counter

# =====================
# Data handling & processing
# =====================
import numpy as np
import pandas as pd
from tqdm import tqdm

# =====================
# Visualization
# =====================
import matplotlib.pyplot as plt
import seaborn as sns

# =====================
# Machine Learning - Core scikit-learn
# =====================
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, ElasticNet
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score,
    root_mean_squared_error, roc_auc_score
)
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.svm import SVC, SVR

# =====================
# Machine Learning - Tree Boosting & advanced
# =====================
import xgboost as xg
import lightgbm as lgb
import catboost

# =====================
# Deep Learning - TensorFlow / Keras
# =====================
import tensorflow as tf
from keras import regularizers
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.optimizers import Adam

# =====================
# Deep Learning - PyTorch
# =====================
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.optim as optim

# =====================
# Imbalanced data handling
# =====================
from imblearn.over_sampling import SMOTE

# =====================
# Optimization / AutoML
# =====================
import optuna

# =====================
# Feature importance & explainability
# =====================
import shap

# =====================
# Self Made Utilities
# =====================
from utils import *

# =====================
# Settings & reproducibility
# =====================
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Libraries successfully loaded. Ready to go!")

Libraries successfully loaded. Ready to go!


In [38]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [39]:
train.head()

,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [ ]:
def create_features(df):
    df_new = df.copy()
    
    df_new['Rhythm_Energy'] = df_new['RhythmScore'] * df_new['Energy']
    df_new['Rhythm_Loudness'] = df_new['RhythmScore'] * df_new['AudioLoudness']
    
    df_new['Duration_Minutes'] = df_new['TrackDurationMs'] / 60000  # Convert to minutes
    df_new['Duration_Energy_Ratio'] = df_new['TrackDurationMs'] / (df_new['Energy'] * 10000 + 1) 
    
    df_new['RhythmScore_Squared'] = df_new['RhythmScore'] ** 2
    df_new['Energy_Squared'] = df_new['Energy'] ** 2
    df_new['Log_Duration'] = np.log1p(df_new['TrackDurationMs'])  # log(1+x) to handle zeros
    
    df_new['Acoustic_Instrumental_Ratio'] = df_new['AcousticQuality'] / (df_new['InstrumentalScore'] + 0.01) 
    df_new['Vocal_Energy'] = df_new['VocalContent'] * df_new['Energy']
    
    df_new['Live_Energy'] = df_new['LivePerformanceLikelihood'] * df_new['Energy']
    df_new['Mood_Rhythm'] = df_new['MoodScore'] * df_new['RhythmScore']
    
    df_new['Audio_Intensity'] = (df_new['Energy'] * np.abs(df_new['AudioLoudness'])) / 10 
    df_new['Performance_Character'] = (df_new['LivePerformanceLikelihood'] + df_new['MoodScore']) / 2
    
    df_new['Energy_Loudness_Ratio'] = df_new['Energy'] / (np.abs(df_new['AudioLoudness']) + 0.01)
    df_new['Rhythm_Duration_Density'] = df_new['RhythmScore'] / df_new['Duration_Minutes']

    # new below

    df_new['Log_RhythmScore'] = np.log1p(df_new['RhythmScore'])
    df_new['Log_Energy'] = np.log1p(df_new['Energy'])
    df_new['Log_AcousticQuality'] = np.log1p(df_new['AcousticQuality'])
    df_new['Log_InstrumentalScore'] = np.log1p(df_new['InstrumentalScore'])
    df_new['Log_VocalContent'] = np.log1p(df_new['VocalContent'])
    df_new['Log_LivePerformanceLikelihood'] = np.log1p(df_new['LivePerformanceLikelihood'])
    df_new['Log_MoodScore'] = np.log1p(df_new['MoodScore'])
    df_new['Log_AudioLoudness'] = np.log1p(np.abs(df_new['AudioLoudness']) + 1)
    
    return df_new

train = create_features(train)
test = create_features(test)

In [41]:
X = train.drop(["id", "BeatsPerMinute"], axis=1)
y = train["BeatsPerMinute"]

X_test = test.drop(["id"], axis=1)

In [42]:
for col in train.columns:
    if train[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        X_test[col] = le.transform(X_test[col])

In [43]:
import lightgbm as lgb
from sklearn.model_selection import KFold

# Use 10-fold stratified cross-validation
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_probs_lgbm = np.zeros(len(X_test))
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training fold {fold + 1}/{n_splits} >>>")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    lightgbm = lgb.LGBMRegressor(
        n_estimators=20000,
        learning_rate=0.06,
        num_leaves=100,
        max_depth=10,
        min_child_samples=9,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.78,
        reg_lambda=3.0,
        random_state=42,
        verbosity=-1,
        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0
    )
    
    lightgbm.fit(
        X_train, 
        y_train, 
        eval_set=[(X_val, y_val)], 
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(period=500)
        ]
    )

    models.append(lightgbm)
    
    # Average predictions across all folds
    y_probs_lgbm += lightgbm.predict(X_test) / n_splits

best_rmse = root_mean_squared_error(y, lightgbm.predict(X))
print(f"\nBest RMSE: {best_rmse:.4f}")

Training fold 1/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[23]	valid_0's l2: 698.886
Training fold 2/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[21]	valid_0's l2: 699.214
Training fold 3/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 703.021
Training fold 4/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[22]	valid_0's l2: 699.95
Training fold 5/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[35]	valid_0's l2: 703.343
Training fold 6/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[15]	valid_0's l2: 703.641
Training fold 7/10 >>>
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:

In [10]:
output_lgbm = pd.DataFrame({
    'id': test.id,
    'BeatsPerMinute': y_probs_lgbm
})

output_lgbm.to_csv('attempt-lightgbm1.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


In [11]:
output_lgbm.head()

,id,BeatsPerMinute
0,524164,119.014855
1,524165,118.575555
2,524166,119.342685
3,524167,119.113536
4,524168,119.407333
